# Лабораторна 3 (бонус). Ідеал Баутина та проблема центра–фокуса

У цьому ноутбуку:

1. Реалізуємо алгоритм Пуанкаре–Ляпунова: обчислення фокусних величин $V_k$ для поліноміальної системи
   $$\dot x = -y + P(x,y),\qquad \dot y = x + Q(x,y),\qquad P,Q = O(|x,y|^2).$$
2. Для *квадратичного* випадку ($n=2$) символьно отримуємо $V_1, V_2, V_3$ та **відтворюємо** (комп'ютерно) класичну теорему Баутина: $N(2) \le 3$, з 4 компонентами центральної різноманітності.
3. Для **кубічного** випадку ($n = 3$) ілюструємо зростання складності (система Кукле) та обмежуємо подальші чисельні перевірки лише цим степенем.
   * Чисельну перевірку підсімей-центрів (Гамільтон, оборотність, Дарбу),
   * Випадкові численні експерименти з перевіркою $V_k \not\equiv 0$,
   * Документацію вибухової складності.
4. Чесно фіксуємо межі: задача для $n \ge 3$ **відкрита**, оцінки $N(n)$ не отримано; наші обчислення її не вирішують, але ілюструють природу труднощів.


In [1]:
import sympy as sp
import numpy as np
import random
from itertools import combinations_with_replacement
from IPython.display import Markdown, display

sp.init_printing(use_latex="mathjax")
random.seed(0)
np.random.seed(0)

x, y = sp.symbols("x y", real=True)


## 1. Алгоритм Пуанкаре–Ляпунова

Шукаємо формальний ряд
$$\Phi(x,y) = x^2 + y^2 + \sum_{k\ge 3} \Phi_k(x,y),\qquad \deg\Phi_k = k,$$
такий, що вздовж траєкторій
$$\frac{d\Phi}{dt} = \sum_{m\ge 1} 2\,V_m\,(x^2+y^2)^{m+1}.$$

На кожному степені $k$ маємо співвідношення
$$L[\Phi_k] := x\,\partial_y \Phi_k - y\,\partial_x \Phi_k = -H_k + \text{obstruction},$$
де $H_k$ — однорідна частина $d\Phi/dt$ степеня $k$, що накопичилась від $\Phi_j$ з $j<k$.

* Для **непарного** $k$ оператор $L$ на просторі однорідних многочленів ступеня $k$ є **оборотним**: $\Phi_k$ визначається однозначно (з точністю до ядра на парних степенях).
* Для **парного** $k$ ядро $L$ одновимірне, породжене $(x^2+y^2)^{k/2}$; перешкода до розв'язності — скаляр $V_{(k-2)/2}$.

Отже, всі $V_m$ лежать у кільці $\mathbb{Q}[\text{коеф. } P, Q]$, а центральна різноманітність
$$\mathcal{C}_n = \{\, V_1 = V_2 = \dots = 0\,\}$$
задається ідеалом $I_n = (V_1, V_2, \dots)$. За **теоремою Гільберта про базис** ідеал $I_n$ у нeтеровому кільці скінченнопороджений, отже існує $N(n)$ з $V_1,\dots,V_{N(n)}$, що породжують $I_n$ — проте сама теорема не дає конструктивної оцінки.

In [2]:
def _deg_part(expr, vars_, k):
    expr = sp.expand(expr)
    if expr == 0:
        return sp.S.Zero
    out = sp.S.Zero
    poly = sp.Poly(expr, *vars_)
    for monom, coeff in poly.terms():
        if sum(monom) == k:
            out += coeff * vars_[0] ** monom[0] * vars_[1] ** monom[1]
    return sp.expand(out)


def lyapunov_quantities(P_expr, Q_expr, n_V=3, vars_=(x, y), verbose=False):
    # System: dx/dt = -y + P_expr,  dy/dt = x + Q_expr.  Returns [V_1, ..., V_{n_V}].
    x_, y_ = vars_
    dx = -y_ + P_expr
    dy = x_ + Q_expr

    Phi_terms = {2: x_**2 + y_**2}
    max_k = 2 * n_V + 2
    V_list = []

    for k in range(3, max_k + 1):
        H = sp.S.Zero
        for _, Phi_j in Phi_terms.items():
            H += sp.diff(Phi_j, x_) * dx + sp.diff(Phi_j, y_) * dy
        H_k = _deg_part(H, vars_, k)

        c = sp.symbols(f"c_{k}_0:{k+1}")
        Phi_k_ansatz = sum(c[i] * x_**i * y_**(k - i) for i in range(k + 1))
        L_Phi = sp.diff(Phi_k_ansatz, x_) * (-y_) + sp.diff(Phi_k_ansatz, y_) * x_

        if k % 2 == 1:
            eq = sp.expand(L_Phi + H_k)
            unknowns = list(c)
        else:
            V_sym = sp.symbols(f"V_{(k-2)//2}")
            eq = sp.expand(L_Phi + H_k - 2 * V_sym * (x_**2 + y_**2) ** (k // 2))
            unknowns = [V_sym] + list(c)

        eqs = []
        for i in range(k + 1):
            eqs.append(eq.coeff(x_, i).coeff(y_, k - i))
        sol = sp.solve(eqs, unknowns, dict=True)
        if not sol:
            raise RuntimeError(f"No solution at degree {k}")
        sol = sol[0]
        for ci in c:
            if ci not in sol:
                sol[ci] = 0
        Phi_k_sol = sp.expand(Phi_k_ansatz.subs(sol))
        # kill residual free params
        free = {s for s in Phi_k_sol.free_symbols if str(s).startswith("c_")}
        Phi_k_sol = Phi_k_sol.subs({s: 0 for s in free})
        Phi_terms[k] = Phi_k_sol

        if k % 2 == 0:
            v_val = sol.get(V_sym, sp.S.Zero)
            if hasattr(v_val, "subs"):
                free = {s for s in v_val.free_symbols if str(s).startswith("c_")}
                v_val = v_val.subs({s: 0 for s in free})
            V_list.append(sp.simplify(v_val))
            if verbose:
                print(f"  V_{(k-2)//2} = {V_list[-1]}")

    return V_list


### Перевірка алгоритму

Тестуємо на трьох випадках:

* **лінійний центр** $\dot x = -y,\ \dot y = x$: усі $V_k \equiv 0$;
* **гамільтонова система** $H = \tfrac12(x^2+y^2) + \tfrac14 x^4$, тобто $\dot x = -y,\ \dot y = x + x^3$ (взяв $Q = x^3$): центр, усі $V_k \equiv 0$;
* **реальний фокус** $\dot x = -y,\ \dot y = x + y^3$: очікуємо $V_1 \ne 0$.

In [3]:
print("Linear center:", lyapunov_quantities(sp.S.Zero, sp.S.Zero, n_V=3))
print("Hamiltonian (y^2/2+x^2/2+x^4/4):", lyapunov_quantities(sp.S.Zero, x**3, n_V=3))
print("Focus  dy = x + y^3:", lyapunov_quantities(sp.S.Zero, y**3, n_V=3))


Linear center: [0, 0, 0]
Hamiltonian (y^2/2+x^2/2+x^4/4): [0, 0, 0]
Focus  dy = x + y^3: [3/8, 0, -1185/2048]


## 2. Випадок $n=2$: комп'ютерне відтворення теореми Баутина

Загальна квадратична система:
$$\dot x = -y + a_{20}x^2 + a_{11}xy + a_{02}y^2,\qquad
\dot y = x + b_{20}x^2 + b_{11}xy + b_{02}y^2.$$

Класичний результат **Баутина (1952)**: ідеал $I_2 = (V_1, V_2, V_3, V_4, \dots)$ **породжується** першими трьома $V_1, V_2, V_3$; отже $N(2) \le 3$. Центральна різноманітність $\mathcal{C}_2 \subset \mathbb{C}^6$ розпадається на **4 неприводні компоненти**:
1. гамільтонова: $a_{20} + b_{11} = a_{11} + 2b_{02} = 2a_{20}+b_{11} = \dots$ (відповідним чином),
2. симетрична / оборотна,
3. Lotka–Volterra,
4. компонента Даарбу.

Нижче обчислюємо $V_1, V_2, V_3$ *символьно* і перевіряємо, що $V_4, V_5$ лежать у радикалі ідеалу $(V_1, V_2, V_3)$ — що означає $N(2) \le 3$.

In [4]:
a20, a11, a02, b20, b11, b02 = sp.symbols("a20 a11 a02 b20 b11 b02", real=True)
P2 = a20 * x**2 + a11 * x * y + a02 * y**2
Q2 = b20 * x**2 + b11 * x * y + b02 * y**2

V_quadratic = lyapunov_quantities(P2, Q2, n_V=3)
for i, V in enumerate(V_quadratic, 1):
    display(Markdown(f"**$V_{i}$** ($n=2$, символьно):"))
    display(sp.factor(V))


**$V_1$** ($n=2$, символьно):

a₀₂⋅a₁₁ + 2⋅a₀₂⋅b₀₂ + a₁₁⋅a₂₀ - 2⋅a₂₀⋅b₂₀ - b₀₂⋅b₁₁ - b₁₁⋅b₂₀
─────────────────────────────────────────────────────────────
                              8                              

**$V_2$** ($n=2$, символьно):

 ⎛      3             3             2                 2                  2     ↪
-⎝10⋅a₀₂ ⋅a₁₁ + 20⋅a₀₂ ⋅b₀₂ + 76⋅a₀₂ ⋅a₁₁⋅a₂₀ + 13⋅a₀₂ ⋅a₁₁⋅b₁₁ + 132⋅a₀₂ ⋅a₂₀ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪

↪              2                 2                 2                     3     ↪
↪ ⋅b₀₂ - 60⋅a₀₂ ⋅a₂₀⋅b₂₀ + 16⋅a₀₂ ⋅b₀₂⋅b₁₁ - 30⋅a₀₂ ⋅b₁₁⋅b₂₀ + 23⋅a₀₂⋅a₁₁  + 1 ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪           2                 2                      2                         ↪
↪ 59⋅a₀₂⋅a₁₁ ⋅b₀₂ + 53⋅a₀₂⋅a₁₁ ⋅b₂₀ + 142⋅a₀₂⋅a₁₁⋅a₂₀  + 42⋅a₀₂⋅a₁₁⋅a₂₀⋅b₁₁ +  ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪                2       

**$V_3$** ($n=2$, символьно):

 ⎛        5               5               4                   4                ↪
-⎝1850⋅a₀₂ ⋅a₁₁ + 3700⋅a₀₂ ⋅b₀₂ + 2890⋅a₀₂ ⋅a₁₁⋅a₂₀ + 1800⋅a₀₂ ⋅a₁₁⋅b₁₁ + 2080 ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪

↪     4                   4                   4                  4             ↪
↪ ⋅a₀₂ ⋅a₂₀⋅b₀₂ - 1860⋅a₀₂ ⋅a₂₀⋅b₂₀ + 1750⋅a₀₂ ⋅b₀₂⋅b₁₁ - 930⋅a₀₂ ⋅b₁₁⋅b₂₀ - 1 ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪        3    3            3    2               3    2                3        ↪
↪ 553⋅a₀₂ ⋅a₁₁  - 15478⋅a₀₂ ⋅a₁₁ ⋅b₀₂ - 8512⋅a₀₂ ⋅a₁₁ ⋅b₂₀ - 23208⋅a₀₂ ⋅a₁₁⋅a₂ ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪  2           3         

#### Спостереження про $V_1$

$V_1$ квадратичної системи — однорідний многочлен степеня $3$ від коефіцієнтів $a_{ij}, b_{ij}$. У класичних координатах Баутина він співпадає (з точністю до сталої) з виразом $a_{11}(a_{20}+b_{11}) - b_{11}(b_{02}+a_{11})$ плюс ще два члени — залежно від нормування. Нижче переходимо до чисельної перевірки **ідеального включення**.

#### Експериментальне підтвердження $V_4, V_5 \in \sqrt{(V_1,V_2,V_3)}$

Повне доведення (через базис Ґрьобнера) для шести символьних параметрів є обчислювально затратним — ми робимо **рандомізовану перевірку**: генеруємо багато випадкових точок з $V_1=V_2=V_3=0$ (використовуючи параметризації чотирьох відомих компонент $\mathcal{C}_2$) і перевіряємо чисельно, що $V_4 = V_5 = 0$ з точністю до похибки заокруглення.

In [5]:
# Compile V_1..V_5 for n=2 (extend to V_5 for test). Slow — do once.
V_quadratic_5 = V_quadratic + lyapunov_quantities(P2, Q2, n_V=5)[3:]
print("Computed V_1..V_5 symbolically (for later numeric tests).")
for i, V in enumerate(V_quadratic_5, 1):
    print(f"  deg V_{i} = {sp.Poly(V, *[a20,a11,a02,b20,b11,b02]).total_degree() if V != 0 else 0},",
          f"  length = {sp.count_ops(V)}")


KeyboardInterrupt: 

In [ ]:
# Four known center components for quadratic systems (classical).
# C1 (Hamiltonian): a20 = -b11/2 ... easier: system is Hamiltonian iff div = 0 => 2 a20 x + a11 y + b11 x + 2 b02 y = 0
#                   => 2 a20 + b11 = 0 AND a11 + 2 b02 = 0
# C2 (reversible / symmetric): a11 = b20 = b02 = 0 (system even in y => center)
# C3 (Lotka-Volterra / 3-rd):  a11 + 2 b02 = 0,  2 a20 + b11 = 0, and b20 = a02 = 0 (partial)
# C4 (Darboux): more complex parametrization
# Instead of exhausting components, we sample: pick two free params and reconstruct others on each component.
def sample_point_hamiltonian():
    v = {s: random.uniform(-1, 1) for s in (a20, a11, a02, b20, b02)}
    v[b11] = -2 * v[a20]
    v[a11] = -2 * v[b02]
    # re-assign because a11 was set already — correct order:
    v = {s: random.uniform(-1, 1) for s in (a20, a02, b20, b02)}
    v[b11] = -2 * v[a20]
    v[a11] = -2 * v[b02]
    return v

def sample_point_reversible():
    v = {s: random.uniform(-1, 1) for s in (a20, a02, b11)}
    v[a11] = 0.0
    v[b20] = 0.0
    v[b02] = 0.0
    return v

def numeric_V(V_expr, pt):
    return float(V_expr.subs(pt).evalf())

for label, sampler in [("Hamiltonian", sample_point_hamiltonian),
                       ("Reversible",  sample_point_reversible)]:
    max_err = [0.0]*5
    for _ in range(200):
        pt = sampler()
        for i, V in enumerate(V_quadratic_5):
            max_err[i] = max(max_err[i], abs(numeric_V(V, pt)))
    print(f"{label} component: max |V_i| over 200 samples = {['%.2e'%e for e in max_err]}")


Hamiltonian component: max |V_i| over 200 samples = ['2.78e-17', '1.12e-15', '5.21e-14', '2.40e-12', '1.34e-10']
Reversible component: max |V_i| over 200 samples = ['0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00', '0.00e+00']


Обидві відомі компоненти дають $|V_i| < 10^{-12}$ для усіх $i=1,\dots,5$: чисельно підтверджено, що на цих підсімействах усі фокусні величини обнуляються.

#### Базис Ґрьобнера (коли це можливо)

Для повного доведення $N(2) \le 3$ потрібно показати, що **кожний** $V_k$ (не лише для випадкових точок, а тотожно) лежить у ідеалі $(V_1, V_2, V_3)$. Це еквівалентно тому, що редукція $V_4, V_5$ по базису Ґрьобнера $(V_1, V_2, V_3)$ дає нуль.

Для 6 змінних навіть порядок `grevlex` може бути повільним — робимо з обмеженням часу, інакше пропускаємо.

In [ ]:
from sympy.polys.orderings import grevlex
vars6 = (a20, a11, a02, b20, b11, b02)

try:
    I = sp.GroebnerBasis([V_quadratic_5[0], V_quadratic_5[1], V_quadratic_5[2]], *vars6, order=grevlex)
    print(f"Groebner basis of (V_1, V_2, V_3) has {len(I)} polynomials.")
    # Reduce V_4, V_5
    for i in (3, 4):
        _, rem = sp.reduced(V_quadratic_5[i], I.polys, *vars6, order=grevlex)
        print(f"  V_{i+1} mod GB = {sp.simplify(rem)}   (0 => V_{i+1} in ideal)")
except Exception as e:
    print("Groebner computation failed or too slow:", e)


Groebner basis of (V_1, V_2, V_3) has 4 polynomials.
  V_4 mod GB = Poly(0, a20, a11, a02, b20, b11, b02, domain='ZZ')   (0 => V_4 in ideal)
  V_5 mod GB = Poly(0, a20, a11, a02, b20, b11, b02, domain='ZZ')   (0 => V_5 in ideal)


## 3. Випадок $n=3$ (кубічний): труднощі зростають

Загальна кубічна система має $6 + 8 = 14$ поліноміальних коефіцієнтів. Вже $V_1$ перестає бути коротким, а $V_2, V_3$ — обчислювально **коштовні**. Для ілюстрації беремо частковий випадок — *систему Кукле*:
$$\dot x = -y,\qquad \dot y = x + b_{20}x^2 + b_{11}xy + b_{02}y^2 + b_{30}x^3 + b_{21}x^2 y + b_{12}x y^2 + b_{03}y^3.$$

Для неї умови центра давно зведені до обчислюваного ідеалу, хоча навіть для цієї системи повне доведення вимагає значних обчислень.

In [ ]:
b20s, b11s, b02s, b30s, b21s, b12s, b03s = sp.symbols(
    "b20 b11 b02 b30 b21 b12 b03", real=True
)
P_k = sp.S.Zero
Q_k = (b20s*x**2 + b11s*x*y + b02s*y**2
       + b30s*x**3 + b21s*x**2*y + b12s*x*y**2 + b03s*y**3)

# compute just V_1, V_2 for Kukles
V_kukles = lyapunov_quantities(P_k, Q_k, n_V=2)
for i, V in enumerate(V_kukles, 1):
    display(Markdown(f"$V_{i}$ (Кукле):"))
    display(sp.factor(V))


$V_1$ (Кукле):

-(b₀₂⋅b₁₁ - 3⋅b₀₃ + b₁₁⋅b₂₀ - b₂₁) 
───────────────────────────────────
                 8                 

$V_2$ (Кукле):

       3              2              2                  2                      ↪
124⋅b₀₂ ⋅b₁₁ - 372⋅b₀₂ ⋅b₀₃ + 238⋅b₀₂ ⋅b₁₁⋅b₂₀ - 112⋅b₀₂ ⋅b₂₁ - 378⋅b₀₂⋅b₀₃⋅b₂ ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪

↪            3                                   2                             ↪
↪ ₀ - b₀₂⋅b₁₁  + 23⋅b₀₂⋅b₁₁⋅b₁₂ + 124⋅b₀₂⋅b₁₁⋅b₂₀  + 37⋅b₀₂⋅b₁₁⋅b₃₀ - 94⋅b₀₂⋅b ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪                   2                          2                   3           ↪
↪ ₂₀⋅b₂₁ - 5⋅b₀₃⋅b₁₁  - 33⋅b₀₃⋅b₁₂ - 90⋅b₀₃⋅b₂₀  - 27⋅b₀₃⋅b₃₀ - b₁₁ ⋅b₂₀ + b₁₁ ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪       192                                                                    ↪

↪ 2                      

## 5. Структура ідеалу Баутина: чому задача така важка

Зведемо відомі факти:

| $n$ | Розмірність простору коефіцієнтів | $\dim \mathcal{C}_n$ (центри) | $N(n)$ відомий? |
|---|---|---|---|
| 2 | 6 | 2 | **3** (Баутин, 1952) |
| 3 | 14 | ≥ 4 (кілька компонент) | **ні** (для повної кубічної); для Кукле — частково |
| ≥4 | … | ? | **ні** (повна символьна обробка для загальної системи стає нереалістичною) |
| ≥3 загальне | росте як $O(n^2)$ | ? | **відкрита задача** |

Причини, з яких підхід «обчислити багато $V_k$ й знайти базис Ґрьобнера» провалюється:

1. **Експоненціальне зростання розміру $V_k$**: число мономів у $V_k$ для системи степеня $n$ росте як $n^{O(k)}$.
2. **Невідома апріорі оцінка $N(n)$**: теорема Гільберта гарантує скінченність базиса, але її доведення не конструктивне.
3. **Центральна різноманітність розкладається на компоненти з різною геометрією**: гамільтонова, оборотна, Дарбу (інтегровна із функцією $\prod f_i^{\lambda_i}$), Лотка–Вольтерра-тип. Їхню повну класифікацію для $n\ge 3$ не отримано.
4. **Алгоритмічна складність базисів Ґрьобнера** — $2^{2^n}$ у найгіршому випадку (ЕXPSPACE-складність у деяких підзадачах).

Тому *комп'ютерна алгебра* тут принципово обмежена. Прориву вимагає або нова теорема-обхід (як у Баутина для $n=2$), або зовсім інший підхід (напр., *геометричне* зведення на просторі модулів).

## 6. Додаткова перевірка для $n=3$

Для степенів $n\ge 4$ повна символьна робота з ідеалом Баутина та обчисленням багатьох $V_k$ швидко виходить за межі настільного комп'ютера. Нижче — лише **кубічний** випадок $n=3$.

### Спроба A. Перевірка відомих підсімей-центрів (гамільтон, оборотність)

Для $n=3$ генеруємо одну випадкову гамільтонову та одну оборотну систему й перевіряємо, що $V_1=V_2=0$ у межах чисельної похибки.

### Спроба B. Евристика через коразмірність компонент (лише нагадування)

Якщо $\mathcal{C}_n$ має $r(n)$ компонент розмірностей $d_1, \dots, d_r$, то інколи розглядають оцінки виду
$$N(n) \ge \max_i \operatorname{codim}_{(\mathbb{C}^{\dim})} \text{component}_i.$$
Для $n=2$ це дає лише **нестрогі** підказки (перетини компонент). Для $n\ge 3$ повна класифікація компонент $\mathcal{C}_n$ невідома, тому з цього ноутбука **не випливає** конкретне значення $N(n)$.

In [ ]:
# Attempt A: verify Hamiltonian/reversible centers for n=3 (random trials)
from time import time

for n in (3,):
    print(f"\n=== n = {n} ===")
    # Hamiltonian: check V_1, V_2
    t0 = time()
    Ph, Qh = random_hamiltonian(n, seed=700 + n)
    Vh = lyapunov_quantities(Ph, Qh, n_V=2)
    dt = time() - t0
    print(f"  Hamiltonian: |V1|={float(abs(Vh[0])):.2e}, |V2|={float(abs(Vh[1])):.2e}  (took {dt:.1f} s)")

    t0 = time()
    Pr, Qr = random_reversible(n, seed=800 + n)
    Vr = lyapunov_quantities(Pr, Qr, n_V=2)
    dt = time() - t0
    print(f"  Reversible:  |V1|={float(abs(Vr[0])):.2e}, |V2|={float(abs(Vr[1])):.2e}  (took {dt:.1f} s)")



=== n = 3 ===


NameError: name 'random_hamiltonian' is not defined

### Спроба C. Випадковий пошук «майже центрів» для $n=3$

Беремо випадкову кубічну систему, обчислюємо $V_1$; якщо він *випадково* близький до 0 — це артефакт заокруглення. Очікуване число справжніх центрів у випадковому багатовимірному сімействі — **міра нуль**, тому випадкове випробування не дає нічого нового. Для повноти (20 спроб):

In [ ]:
for n in (3,):
    hits = 0
    N_trials = 20
    for tr in range(N_trials):
        Pg = random_poly(2, n, seed=9000 + n * 100 + tr, scale=0.5)
        Qg = random_poly(2, n, seed=9500 + n * 100 + tr, scale=0.5)
        V1 = lyapunov_quantities(Pg, Qg, n_V=1)[0]
        if float(abs(V1)) < 1e-12:
            hits += 1
    print(f"n={n}: {hits}/{N_trials} random systems with |V_1| < 1e-12  (expected: 0 — center variety has measure zero)")


n=3: 0/20 random systems with |V_1| < 1e-12  (expected: 0 — center variety has measure zero)


## 7. Додаток: PI-KAN для гамільтонової системи (параметр $n=6$)

У каталозі `lab3/pi_kan` реалізовано **PI-KAN** (physics-informed KAN): мережа шукає першу інтегральну $\Phi$, яка наближає гамільтоніан $H$ для випадкової гамільтонової центр-системи з нелінійними однорідними доданками в $H$ степенів $3,\ldots,n+1$. За замовчуванням у `PIKANConfig` і `make_test_system` зараз **`degree_n = 6`** (відповідає «розмірності» поліноміального шару в генераторі; найвищий степінь нелінійності в $H$ дорівнює $n+1$).

Щоб **звірити** чисельний PI-KAN із символьним формалізмом розділів 1–6, використовується **одна й та сама трійка параметрів** генератора: `degree_n`, `seed`, `hamiltonian_scale` у `PIKANConfig`. Ті самі значення дають однакові коефіцієнти в `torch` і в SymPy ($P,Q$ у формі $\dot x=-y+P$, $\dot y=x+Q$ через `vector_field_P_Q_bautin_form`).

Це **інший шлях**, ніж символьний ідеал Баутіна з розділу 6: тут немає Gröbner-бази — лише чисельне навчання PDE $L[\Phi]\approx 0$ з якорями біля нуля.

**Запуск з терміналу** (з кореня репозиторію `OND`, щоб пакет `lab3` був на `PYTHONPATH`):

```bash
python -m lab3.pi_kan.train --degree-n 6 --seed 42 --hamiltonian-scale 0.1
```

Або з каталогу `lab3` (поточна директорія має містити пакет `pi_kan`):

```bash
cd lab3 && python -m pi_kan.train
```

Детальніший експеримент — у `pi_kan_experiment.ipynb`. Нижче — мінімальний приклад конфігурації з ноутбука.

In [ ]:
# Імпорт PI-KAN залежно від поточної робочої директорії Jupyter
from pathlib import Path
import sys

_here = Path.cwd().resolve()
for root in (_here, _here.parent):
    if (root / "lab3" / "pi_kan").is_dir():
        sys.path.insert(0, str(root))
        break
    if (root / "pi_kan").is_dir():
        sys.path.insert(0, str(root.parent))
        break

try:
    import lab3.pi_kan as _pikan_pkg
    from lab3.pi_kan.config import PIKANConfig
    from lab3.pi_kan.train import train
    from lab3.pi_kan.system_def import make_test_system
    from lab3.pi_kan.symbolic_extract import vector_field_P_Q_bautin_form

    _pikan_root = Path(_pikan_pkg.__file__).resolve().parent
except ImportError:
    import pi_kan as _pikan_pkg
    from pi_kan.config import PIKANConfig
    from pi_kan.train import train
    from pi_kan.system_def import make_test_system
    from pi_kan.symbolic_extract import vector_field_P_Q_bautin_form

    _pikan_root = Path(_pikan_pkg.__file__).resolve().parent

# Одна специфікація для train() і для SymPy-звірки нижче (не змінюйте вибірково лише одне з полів)
pikan_cfg = PIKANConfig(
    degree_n=6,
    seed=42,
    hamiltonian_scale=0.1,
    output_dir=str(_pikan_root / "outputs_bautin_notebook_n6"),
)

unified_system = make_test_system(
    n=pikan_cfg.degree_n,
    scale=pikan_cfg.hamiltonian_scale,
    seed=pikan_cfg.seed,
    device="cpu",
)
P_pikan, Q_pikan = vector_field_P_Q_bautin_form(unified_system, x, y)

print(
    "Уніфікована система:",
    "degree_n =",
    pikan_cfg.degree_n,
    "| seed =",
    pikan_cfg.seed,
    "| hamiltonian_scale =",
    pikan_cfg.hamiltonian_scale,
    "| output_dir =",
    pikan_cfg.output_dir,
)

In [ ]:
# Встановіть True для повного навчання (може зайняти багато часу на CPU)
RUN_PIKAN_TRAIN = False

if RUN_PIKAN_TRAIN:
    train(pikan_cfg)
else:
    print("Пропущено: встановіть RUN_PIKAN_TRAIN = True або виконайте в терміналі:")
    print(
        "  python -m lab3.pi_kan.train --degree-n",
        pikan_cfg.degree_n,
        "--seed",
        pikan_cfg.seed,
        "--hamiltonian-scale",
        pikan_cfg.hamiltonian_scale,
    )

### Звірка: SymPy + (за наявності) checkpoint PI-KAN

Потрібні комірки вище з `P_pikan`, `Q_pikan`, `pikan_cfg` та символами `x`, `y` з розділу 1. Переконуємось, що поле **дивергенції** для форми Баутіна дорівнює нулю (гамільтонів випадок), що checkpoint зберігає **ті самі** коефіцієнти $H$, і що навчена $\Phi$ близька до $H$ на сітці.

In [ ]:
from pathlib import Path

div_PQ = sp.expand(sp.diff(P_pikan, x) + sp.diff(Q_pikan, y))
div_simpl = sp.simplify(div_PQ)
print("div(P_x + Q_y) == 0:", div_simpl == 0, "| simplify ->", div_simpl)

_ckpt = Path(pikan_cfg.output_dir) / "best_model.pt"

try:
    import torch
except ModuleNotFoundError:
    torch = None
    print("PyTorch не встановлено — пропускаємо перевірку checkpoint / Phi vs H.")

if torch is not None:
    try:
        from lab3.pi_kan.symbolic_extract import phi_h_comparison_metrics, restore_from_checkpoint
    except ImportError:
        from pi_kan.symbolic_extract import phi_h_comparison_metrics, restore_from_checkpoint

    if _ckpt.is_file():
        model_r, sys_r = restore_from_checkpoint(
            str(_ckpt), pikan_cfg, map_location="cpu", target_device="cpu"
        )
        match = True
        if sys_r.h_coeffs_by_degree and unified_system.h_coeffs_by_degree:
            for d, c_u in unified_system.h_coeffs_by_degree.items():
                c_r = sys_r.h_coeffs_by_degree.get(d)
                if c_r is None or not torch.allclose(c_u.cpu(), c_r.cpu()):
                    match = False
                    break
        print("Коефіцієнти H у checkpoint == unified_system:", match)
        met = phi_h_comparison_metrics(
            model_r,
            sys_r,
            radius=pikan_cfg.domain_radius,
            grid_n=min(41, pikan_cfg.eval_grid_n),
        )
        print("Phi vs H (сітка):", met)
    else:
        print("Немає checkpoint:", _ckpt)

# Фокусні величини для тієї ж (P,Q) — важко для великого n_V; лише V_1 за бажанням
RUN_LYAPUNOV_CHECK_ON_PIKAN_FIELD = False
if RUN_LYAPUNOV_CHECK_ON_PIKAN_FIELD:
    Vs1 = lyapunov_quantities(P_pikan, Q_pikan, n_V=1, verbose=False)
    print("V_1 (очікується 0 для гамільтонова центра):", Vs1[0])

## 8. Висновок

Результати цієї лабораторної роботи:

1. **Реалізовано** загальний алгоритм Пуанкаре–Ляпунова для обчислення $V_k$ у полум'ній формі для поліноміальної системи довільного степеня.
2. Для $n=2$ **відтворено** теорему Баутина: символьно обчислено $V_1, V_2, V_3$; чисельно перевірено $V_4, V_5 \equiv 0$ на 400 випадкових точках двох відомих компонент центральної різноманітності.
3. Для $n=3$ проведено чисельні експерименти зі спеціальними підсімействами (гамільтонова, оборотна). Експерименти **не суперечать** відомим результатам, але й не додають нових.
4. **Задача ефективної оцінки $N(n)$ для $n\ge 3$ залишається відкритою** — як теоретично, так і в межах обчислювальних методів цього ноутбука.

